<a href="https://colab.research.google.com/github/manolasayed0-ai/Ai_house_assistant/blob/main/project_linuxa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Imports And Downloads**

In [ ]:
!pip install langchain langchain-google-genai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.5 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [langchain-google-genai]
ERROR: pip's dependency resolver d

In [ ]:
import os                                                                  #التعامل مع نظام التشغيل و ال environment varaibles
import json
from langchain_core.prompts import PromptTemplate                          #عمل برومبت منظم
from langchain_core.output_parsers import PydanticOutputParser             #تتأكد إن ال output ماشي مع ال schema  اللي حاطينها
from pydantic import Field , BaseModel                                     #BaseModel: كلاس بنبني عليها الـSchema بتاعتنا
                                                                           #Field: بنستخدمها عشان نحدد وصفًا أو شروطًا للبيانات
from typing import Optional
from langchain_google_genai import ChatGoogleGenerativeAI                  #دي اللي هتخلينا نتعامل مع Gemini من خلال Google API
from google.colab import userdata

##**info class & Schema**

In [ ]:
class Command(BaseModel):                                                  #Command: اسم الكلاس اللي بيمثل الأمر المطلوب من الـOS
    action: str = Field(
        description="The action to perform"
    )
    target: Optional[str] = Field(
        default=None,
        description="The target of the action"
    )
    name: Optional[str] = Field(
        default=None,
        description="The name of the item"
    )
    path: Optional[str] = Field(
        default=None,
        description="The file or folder path"
    )
    url: Optional[str] = Field(
        default=None,
        description="The URL to open")
    value: Optional[int] = Field(
        default=None,
        description="Numeric value for the action"
    )

class CommandList(BaseModel):
    commands: list[Command]


##**Model + Prompt**

In [ ]:
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
  try:
        api_key = userdata.get("GOOGLE_API_KEY")
  except (ImportError, Exception):
        api_key = None

if not api_key:
    raise ValueError("API key is missing!")                               #os.getenv() بتقرأ قيمة متخزنة في Environment Variables.
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.0,
    google_api_key=api_key)


parser = PydanticOutputParser(                                              #يحول رد المودل بحيث يبقى مطابق لل الإسكيما
pydantic_object=CommandList)

prompt = PromptTemplate(                                                    #الـ Prompt هو التعليمات اللي هنقدمها للموديل
    template=(
        "Understand the user's operating system command "
        "and extract the required structured information.\n"
        "If a field is missing, return null for it.\n\n"
        "{format_instructions}\n\n"
        "User command:\n{text}"
        "- If the user asks to open a website or an online service, provide its official URL when known."
        "- If the user asks to open a locally installed application, return null for the URL."
        "- Do not invent URLs."
        "- If the URL is unknown, return null."
    ),
    input_variables=["text"],
    partial_variables={
        "format_instructions": parser.get_format_instructions()             #parser.get_format_instructions(): بتجيب تعليمات التنسيق اللي الـ Parser بيولّدها بناءً على الـ Schema.
    },
)

In [ ]:
def extract_information(text: str):
    chain = prompt | llm | parser                                         #بنوصلهم ببعض بالترتيب

    try:
        output: CommandList = chain.invoke({"text": text})                    #chain.invoke() بتشغّل السلسلة وبتبعت البيانات للمراحل اللي جهّزناها.
        print(output.model_dump_json(indent=2))                           #model_dump_json() بتحوّل كائن Pydantic إلى نص بصيغة JSON
                                                                          #indent=2 بتنظّم شكل النتيجة بمسافات عشان تبقى سهلة القراءة
        return output

    except Exception as e:
        print(f"Error: {e}")
        return None



##**Test case**

In [ ]:
print("================ TEST CASE 1 ================")

result = extract_information(
    "إفتحلي التيكتوك بعدين ادخل ع جهات الإتصال هاتلي رقم منال السيد"
)



================ TEST CASE 1 ================
{
  "commands": [
    {
      "action": "open",
      "target": "application",
      "name": "تيكتوك",
      "path": null,
      "url": "https://www.tiktok.com",
      "value": null
    },
    {
      "action": "open",
      "target": "application",
      "name": "جهات الإتصال",
      "path": null,
      "url": null,
      "value": null
    },
    {
      "action": "get",
      "target": "contact",
      "name": "منال السيد",
      "path": null,
      "url": null,
      "value": null
    }
  ]
}


In [ ]:
result = extract_information(
    "Create a file named notes.txt in Documents"
)


{
  "commands": [
    {
      "action": "create",
      "target": "file",
      "name": "notes.txt",
      "path": "Documents",
      "url": null,
      "value": null
    }
  ]
}


In [ ]:
result = extract_information(
 " إفتحلي الفيسبوك"
)


{
  "commands": [
    {
      "action": "open",
      "target": "website",
      "name": "Facebook",
      "path": null,
      "url": "https://www.facebook.com",
      "value": null
    }
  ]
}


In [ ]:
result = extract_information("افتح TikTok وطي الصوت لـ 70%")

{
  "commands": [
    {
      "action": "open",
      "target": "application",
      "name": "TikTok",
      "path": null,
      "url": "https://www.tiktok.com",
      "value": null
    },
    {
      "action": "set_volume",
      "target": "system",
      "name": null,
      "path": null,
      "url": null,
      "value": 70
    }
  ]
}


In [ ]:
result = extract_information("انشألي فولدر اسمه Download")

{
  "commands": [
    {
      "action": "create",
      "target": "folder",
      "name": "Download",
      "path": null,
      "url": null,
      "value": null
    }
  ]
}
